# NIR White Lagkitan Corn (Binary-Class)

In [1]:
import pandas as pd
import numpy as np

## Load Data

In [2]:
df_raw_data = pd.read_csv('appendix_raw_dataset2.csv')
df_raw_data

,Class,730nm,760nm,810nm,860nm,900nm,940nm,Brix (%),Sample No.
0,Not Sweet,223.095184,153.247070,308.192169,1929.094238,88.592545,43.446762,6.4,1
1,Not Sweet,225.528946,155.787079,315.297485,1996.374756,90.668930,43.446762,6.4,1
2,Not Sweet,213.360123,154.940414,306.415863,2071.775391,86.516159,43.446762,6.4,1
3,Not Sweet,197.946274,147.320389,278.882843,2032.334961,81.671249,38.509628,6.4,1
4,Not Sweet,247.432846,177.800476,350.823944,2400.057617,90.668930,40.484482,6.4,1
...,...,...,...,...,...,...,...,...,...
4155,Sweet,244.999069,174.413788,367.699036,2205.176270,131.504562,51.346172,9.4,130
4156,Sweet,244.999069,174.413788,352.600281,2121.655518,121.814751,51.346172,9.4,130
4157,Sweet,249.866608,175.260468,365.034546,2181.976074,128.043915,55.295879,9.4,130
4158,Sweet,238.509033,171.027115,349.047638,2133.255615,120.430489,52.333599,9.4,130


## Preprocessing

In [3]:
feature_cols = ['730nm','760nm','810nm','860nm','900nm','940nm']

### Outlier Removal

In [4]:
def filter_scans_stage1(df, keep_ratio_fallback=0.85):
    filtered_parts = []

    for sample_id, group in df.groupby('Sample No.'):
        X = group[feature_cols].values

        centroid = X.mean(axis=0)

        dists = np.linalg.norm(X - centroid, axis=1)

        med = np.median(dists)
        mad = np.median(np.abs(dists - med))

        threshold = med + 1.5 * mad

        mask = dists <= threshold

        if mask.sum() < len(group) * 0.5:
            cutoff = np.quantile(dists, keep_ratio_fallback)
            mask = dists <= cutoff

        filtered_group = group[mask]
        filtered_parts.append(filtered_group)

    return pd.concat(filtered_parts, ignore_index=True)

df_stage1 = filter_scans_stage1(df_raw_data)

### SNV

In [5]:
df_snv = df_stage1.copy()

def snv_row(x):
    x = x.astype(float)
    mean = x.mean()
    std = x.std(ddof=1)

    if std < 1e-8:
        return x * np.nan

    return (x - mean) / std

df_snv[feature_cols] = df_snv[feature_cols].apply(
    snv_row, axis=1, result_type='expand'
)

df_snv = df_snv.dropna()

### Median Aggregation

In [6]:
def stage2_median_aggregation(df):

    df_agg = df.groupby('Sample No.')[feature_cols].median().reset_index()

    class_map = df.groupby('Sample No.')['Class'].first().reset_index()

    df_final = pd.merge(df_agg, class_map, on='Sample No.')

    return df_final

df_stage2 = stage2_median_aggregation(df_snv)

## Data Preparation

### Selecting Features (X) and Targets (y)

In [10]:
X = df_stage2[feature_cols]
y = df_stage2['Class']

### Encode Labels

In [13]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_encoded = le.fit_transform(y)

In [14]:
dict(zip(le.classes_, le.transform(le.classes_)))

{'Not Sweet': np.int64(0), 'Sweet': np.int64(1)}

### Data Split

In [15]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(X, y_encoded, test_size=0.3, stratify=y_encoded, random_state=42)
X_val, X_test, y_val , y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

## Optuna Hyperparameter Tuning

In [16]:
import optuna
from sklearn.model_selection import cross_val_score, RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from xgboost import XGBClassifier

# For reproducibility
sampler = optuna.samplers.TPESampler(seed=42)
inner_cv = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=42)

# Tune to penalize instability
alpha = 0.5
stability_std_cap = 0.02

C:\Users\Lorine Jane\PycharmProjects\Thesis 2nd Final Testing\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Logistic Regression

In [17]:
def objective_lr(trial):
    params = {
        'C': trial.suggest_float('C', 0.05, 3.0, log=True),

        'penalty': 'l2',

        'solver': trial.suggest_categorical(
            'solver', ['liblinear', 'lbfgs']
        ),

        'class_weight': trial.suggest_categorical(
            'class_weight', ['balanced', None]
        ),

        'max_iter': 5000,
        'tol': trial.suggest_float('tol', 1e-5, 1e-3, log=True),
        'random_state': 42
    }

    model = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(**params))
    ])

    score = cross_val_score(
        model,
        X_train,
        y_train,
        cv=inner_cv,
        scoring='roc_auc'
    )

    mean_score = score.mean()
    std_score = score.std()

    return mean_score - alpha * min(std_score, stability_std_cap)

In [18]:
study_lr = optuna.create_study(direction='maximize', sampler=sampler)
study_lr.optimize(objective_lr, n_trials=100)

study_lr.best_params, study_lr.best_value

[I 2026-07-28 18:48:57,729] A new study created in memory with name: no-name-50a76009-f66a-43ec-a13a-7a0089e964a3
C:\Users\Lorine Jane\PycharmProjects\Thesis 2nd Final Testing\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\Lorine Jane\PycharmProjects\Thesis 2nd Final Testing\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Use

({'C': 0.5168899683396826,
  'solver': 'liblinear',
  'class_weight': None,
  'tol': 0.0007145407933214536},
 0.5873333333333333)

### Random Forest

In [19]:
def objective_rf(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 150, 350),

        'max_depth': trial.suggest_int('max_depth', 2, 5),

        'max_features': trial.suggest_categorical(
            'max_features', ['sqrt', 0.5]
        ),

        'min_samples_split': trial.suggest_int('min_samples_split', 5, 15),

        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 2, 8),

        'max_leaf_nodes': trial.suggest_int('max_leaf_nodes', 8, 25),

        'class_weight': trial.suggest_categorical(
            'class_weight', ['balanced', None]
        ),

        'bootstrap': True,
        'n_jobs': -1,
        'random_state': 42
    }

    model = Pipeline([
        ('clf', RandomForestClassifier(**params))
    ])

    score = cross_val_score(
        model,
        X_train,
        y_train,
        cv=inner_cv,
        scoring='roc_auc'
    )

    mean_score = score.mean()
    std_score = score.std()

    return mean_score - alpha * min(std_score, stability_std_cap)

In [20]:
study_rf = optuna.create_study(direction='maximize', sampler=sampler)
study_rf.optimize(objective_rf, n_trials=100)

study_rf.best_params, study_rf.best_value

[I 2026-07-28 18:49:49,879] A new study created in memory with name: no-name-864a451c-3d18-4145-a9eb-335a0d2bbcd3
[I 2026-07-28 18:50:10,193] Trial 0 finished with value: 0.4943333333333334 and parameters: {'n_estimators': 228, 'max_depth': 3, 'max_features': 'sqrt', 'min_samples_split': 8, 'min_samples_leaf': 5, 'max_leaf_nodes': 10, 'class_weight': 'balanced'}. Best is trial 0 with value: 0.4943333333333334.
[I 2026-07-28 18:50:37,906] Trial 1 finished with value: 0.482 and parameters: {'n_estimators': 348, 'max_depth': 5, 'max_features': 'sqrt', 'min_samples_split': 13, 'min_samples_leaf': 6, 'max_leaf_nodes': 21, 'class_weight': 'balanced'}. Best is trial 0 with value: 0.4943333333333334.
[I 2026-07-28 18:50:57,902] Trial 2 finished with value: 0.5346666666666666 and parameters: {'n_estimators': 222, 'max_depth': 2, 'max_features': 'sqrt', 'min_samples_split': 8, 'min_samples_leaf': 2, 'max_leaf_nodes': 13, 'class_weight': None}. Best is trial 2 with value: 0.5346666666666666.
[I 2

({'n_estimators': 150,
  'max_depth': 2,
  'max_features': 0.5,
  'min_samples_split': 14,
  'min_samples_leaf': 3,
  'max_leaf_nodes': 13,
  'class_weight': None},
 0.5556666666666666)

### Linear Discriminant Analysis

In [21]:
def objective_lda(trial):
    solver = trial.suggest_categorical('solver', ['lsqr', 'eigen'])

    params = {
        'solver': solver,

        'shrinkage': trial.suggest_float(
            'shrinkage',
            0.05, 0.8,
            log=True
        ),

        'store_covariance': False
    }

    model = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LinearDiscriminantAnalysis(**params))
    ])

    score = cross_val_score(
        model,
        X_train,
        y_train,
        cv=inner_cv,
        scoring='roc_auc'
    )

    mean_score = score.mean()
    std_score = score.std()

    return mean_score - alpha * min(std_score, stability_std_cap)

In [22]:
study_lda = optuna.create_study(direction='maximize', sampler=sampler)
study_lda.optimize(objective_lda, n_trials=100)

study_lda.best_params, study_lda.best_value

[I 2026-07-28 19:07:49,734] A new study created in memory with name: no-name-521a3573-b087-4c1a-b32b-359add9b650d
[I 2026-07-28 19:07:50,083] Trial 0 finished with value: 0.5809999999999998 and parameters: {'solver': 'lsqr', 'shrinkage': 0.07472067993851027}. Best is trial 0 with value: 0.5809999999999998.
[I 2026-07-28 19:07:50,410] Trial 1 finished with value: 0.5856666666666666 and parameters: {'solver': 'eigen', 'shrinkage': 0.0978213368435106}. Best is trial 1 with value: 0.5856666666666666.
[I 2026-07-28 19:07:50,698] Trial 2 finished with value: 0.5856666666666666 and parameters: {'solver': 'eigen', 'shrinkage': 0.09663047618134958}. Best is trial 1 with value: 0.5856666666666666.
[I 2026-07-28 19:07:50,995] Trial 3 finished with value: 0.5709999999999998 and parameters: {'solver': 'lsqr', 'shrinkage': 0.28863041253831084}. Best is trial 1 with value: 0.5856666666666666.
[I 2026-07-28 19:07:51,240] Trial 4 finished with value: 0.5823333333333333 and parameters: {'solver': 'lsqr'

({'solver': 'eigen', 'shrinkage': 0.0978213368435106}, 0.5856666666666666)

### Support Vector Machine

In [23]:
def objective_svc(trial):
    kernel = trial.suggest_categorical('kernel', ['linear', 'rbf'])

    C = trial.suggest_float('C', 0.05, 5.0, log=True)

    params = {
        'C': C,
        'kernel': kernel,
        'class_weight': trial.suggest_categorical(
            'class_weight', ['balanced', None]
        ),
    }

    if kernel == 'rbf':
        params['gamma'] = trial.suggest_float(
            'gamma',
            1e-4,
            0.01,
            log=True
        )

    model = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SVC(**params))
    ])

    score = cross_val_score(
        model,
        X_train,
        y_train,
        cv=inner_cv,
        scoring='roc_auc'
    )

    mean_score = score.mean()
    std_score = score.std()

    return mean_score - alpha * min(std_score, stability_std_cap)

In [24]:
study_svc = optuna.create_study(direction='maximize', sampler=sampler)
study_svc.optimize(objective_svc, n_trials=100)

study_svc.best_params, study_svc.best_value

[I 2026-07-28 19:08:15,913] A new study created in memory with name: no-name-d99bfee9-4644-4993-be8c-a11423d8ae5e
[I 2026-07-28 19:08:16,192] Trial 0 finished with value: 0.5640000000000001 and parameters: {'kernel': 'linear', 'C': 3.5347380986089947, 'class_weight': 'balanced'}. Best is trial 0 with value: 0.5640000000000001.
[I 2026-07-28 19:08:16,426] Trial 1 finished with value: 0.5143333333333333 and parameters: {'kernel': 'rbf', 'C': 0.644720719653709, 'class_weight': 'balanced', 'gamma': 0.00015353434324857047}. Best is trial 0 with value: 0.5640000000000001.
[I 2026-07-28 19:08:16,659] Trial 2 finished with value: 0.5299999999999999 and parameters: {'kernel': 'rbf', 'C': 0.9229388317394318, 'class_weight': None, 'gamma': 0.0028308141496998056}. Best is trial 0 with value: 0.5640000000000001.
[I 2026-07-28 19:08:16,950] Trial 3 finished with value: 0.5599999999999999 and parameters: {'kernel': 'linear', 'C': 1.8143501127770325, 'class_weight': 'balanced'}. Best is trial 0 with v

({'kernel': 'linear', 'C': 0.10440890541682227, 'class_weight': 'balanced'},
 0.5753333333333333)

### XGBoost

In [25]:
def objective_xgb(trial):
    params = {
        'max_depth': trial.suggest_int('max_depth', 2, 3),

        'learning_rate': trial.suggest_float(
            'learning_rate',
            0.02,
            0.05,
            log=True
        ),

        'n_estimators': trial.suggest_int('n_estimators', 80, 180),

        'min_child_weight': trial.suggest_float('min_child_weight', 3.0, 10.0),

        'gamma': trial.suggest_float('gamma', 0.1, 0.3),

        'subsample': trial.suggest_float('subsample', 0.8, 1.0),

        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.8, 1.0),

        'reg_lambda': trial.suggest_float('reg_lambda', 1.0, 10.0, log=True),

        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 0.1),

        'max_delta_step': trial.suggest_int('max_delta_step', 0, 2),

        'random_state': 42,
        'n_jobs': -1,
        'eval_metric': 'logloss'
    }

    model = Pipeline([
        ('clf', XGBClassifier(**params))
    ])

    score = cross_val_score(
        model,
        X_train,
        y_train,
        cv=inner_cv,
        scoring='roc_auc')

    mean_score = score.mean()
    std_score = score.std()

    return mean_score - alpha * min(std_score, stability_std_cap)

In [26]:
study_xgb = optuna.create_study(direction='maximize', sampler=sampler)
study_xgb.optimize(objective_xgb, n_trials=100)

study_xgb.best_params, study_xgb.best_value

[I 2026-07-28 19:08:40,104] A new study created in memory with name: no-name-651be8c1-9768-4bd9-b04d-fe7a64908b84
[I 2026-07-28 19:08:41,486] Trial 0 finished with value: 0.4858333333333334 and parameters: {'max_depth': 3, 'learning_rate': 0.025867862988964596, 'n_estimators': 82, 'min_child_weight': 7.518306071350175, 'gamma': 0.13542213588140978, 'subsample': 0.9880917168705828, 'colsample_bytree': 0.9907857154005175, 'reg_lambda': 8.219859422471128, 'reg_alpha': 0.03701587002554444, 'max_delta_step': 0}. Best is trial 0 with value: 0.4858333333333334.
[I 2026-07-28 19:08:43,090] Trial 1 finished with value: 0.4875 and parameters: {'max_depth': 3, 'learning_rate': 0.02960885405829663, 'n_estimators': 177, 'min_child_weight': 9.74533983962477, 'gamma': 0.270601891093472, 'subsample': 0.8588897784139171, 'colsample_bytree': 0.8770195457203851, 'reg_lambda': 7.0980110566411465, 'reg_alpha': 0.031692200515627766, 'max_delta_step': 0}. Best is trial 1 with value: 0.4875.
[I 2026-07-28 19:

({'max_depth': 2,
  'learning_rate': 0.04772753454707194,
  'n_estimators': 135,
  'min_child_weight': 5.708054401119242,
  'gamma': 0.29256771005742016,
  'subsample': 0.9879709827503215,
  'colsample_bytree': 0.9936378122095748,
  'reg_lambda': 1.4992089279125758,
  'reg_alpha': 0.07944657116549703,
  'max_delta_step': 2},
 0.5536666666666666)

## Validation Set Metrics

In [27]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

In [28]:
models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(**study_lr.best_params))
    ]),
    'Random Forest': Pipeline([
        ('clf', RandomForestClassifier(**study_rf.best_params))
    ]),
    'Linear Discriminant Analysis': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LinearDiscriminantAnalysis(**study_lda.best_params))
    ]),
    'Support Vector Machine': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SVC(**study_svc.best_params))
    ]),
    'XGBoost': Pipeline([
        ('clf', XGBClassifier(**study_xgb.best_params))
    ])
}

In [29]:
val_predictions = {}
val_metrics = {}
val_reports ={}

for model_name, model in models.items():

    model.fit(X_train, y_train)

    val_predictions[model_name] = model.predict(X_val)

    val_metrics[model_name] = {
        'Accuracy': accuracy_score(y_val, val_predictions[model_name]),
        'Precision': precision_score(y_val, val_predictions[model_name], average='weighted', zero_division=0),
        'Recall': recall_score(y_val, val_predictions[model_name], average='weighted', zero_division=0),
        'F1': f1_score(y_val, val_predictions[model_name], average='weighted', zero_division=0),
    }

    val_reports[model_name] = classification_report(y_val, val_predictions[model_name], zero_division=0)

### Summary Metrics

In [30]:
df_val_metrics = pd.DataFrame(val_metrics)
df_val_metrics.T.sort_values(by='Accuracy', ascending=False)

,Accuracy,Precision,Recall,F1
Random Forest,0.789474,0.854251,0.789474,0.782297
XGBoost,0.736842,0.759109,0.736842,0.727689
Support Vector Machine,0.578947,0.587321,0.578947,0.576608
Linear Discriminant Analysis,0.473684,0.473684,0.473684,0.473684
Logistic Regression,0.421053,0.423392,0.421053,0.421053


### Classification Report

In [31]:
for model_names, reports in val_reports.items():
    print(f'{model_names}:\n {reports} \n -------------------------------------------------------')

Logistic Regression:
               precision    recall  f1-score   support

           0       0.44      0.40      0.42        10
           1       0.40      0.44      0.42         9

    accuracy                           0.42        19
   macro avg       0.42      0.42      0.42        19
weighted avg       0.42      0.42      0.42        19
 
 -------------------------------------------------------
Random Forest:
               precision    recall  f1-score   support

           0       1.00      0.60      0.75        10
           1       0.69      1.00      0.82         9

    accuracy                           0.79        19
   macro avg       0.85      0.80      0.78        19
weighted avg       0.85      0.79      0.78        19
 
 -------------------------------------------------------
Linear Discriminant Analysis:
               precision    recall  f1-score   support

           0       0.50      0.50      0.50        10
           1       0.44      0.44      0.44         

## Test Set Metrics

In [32]:
X_combined = np.concatenate([X_train, X_val], axis=0)
y_combined = np.concatenate([y_train, y_val], axis=0)

In [33]:
final_model = XGBClassifier(**study_xgb.best_params)
final_model.fit(X_combined, y_combined)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.9936378122095748
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegres

### Summary Metrics

In [34]:
test_predictions = final_model.predict(X_test)

test_metrics = {
    'Model': 'XGBoost',
    'Accuracy': accuracy_score(y_test, test_predictions),
    'Precision': precision_score(y_test, test_predictions, average='weighted', zero_division=0),
    'Recall': recall_score(y_test, test_predictions, average='weighted', zero_division=0),
    'F1': f1_score(y_test, test_predictions, average='weighted', zero_division=0),
}

In [35]:
df_test_metrics = pd.DataFrame([test_metrics])
df_test_metrics

,Model,Accuracy,Precision,Recall,F1
0,XGBoost,0.85,0.884615,0.85,0.846547


### Classification Report

In [36]:
print(classification_report(y_test, test_predictions))

              precision    recall  f1-score   support

           0       0.77      1.00      0.87        10
           1       1.00      0.70      0.82        10

    accuracy                           0.85        20
   macro avg       0.88      0.85      0.85        20
weighted avg       0.88      0.85      0.85        20

